<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 4 &middot; Make it work</div><div style="font-size:15px;color:#cfd4ff;">The hill climb &middot; about 12 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 4. Make it work (v0 to v1)

**Time:** about 12 minutes. Now the climb begins.

**The question we're starting with:** how do I find out what's wrong with my agent without reading every single answer?

Here's the plan for this module:

1. Watch v0 stumble on a few questions.
2. Use the SDK to give the agent better instructions. That's **v1**.
3. Send it a set of practice questions so it leaves **traces**.
4. Ask Foundry for **insights**: a plain-English list of where the agent is weak, and why.

The big idea: **we don't guess what's wrong. We let the evidence tell us.** Insights show us which way is up before we take a step.

| Our name | Foundry version | What changed |
|---|---|---|
| v0 | Saved by setup | Generic instructions and the manuals |
| v1 | Saved below | Today: better instructions, written through the SDK |


---

## 4.1 Where are we starting?

In [1]:
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the LAB663 repository.")
sys.path.insert(0, str(root / "src"))
import lab

project_client, openai_client = lab.connect()
state = lab.load_state()
if "v0_version" not in state:
    raise RuntimeError("Run setup and notebook 01 first so v0 is pinned.")
practice = lab.load_cases("practice")
print(f"Connected. {len(practice)} practice questions ready.")

Connected. 12 practice questions ready.


## 4.2 How bad is v0, really?

Let's ask the plain v0 agent three questions from our practice set and read the answers *critically*. Each one is aimed at a different way an agent can let a customer down.

Before you run it, **predict**: which one is v0 most likely to get wrong?

In [2]:
# Three practice questions that each probe a different weakness.
for case_id in ["pr-03", "pr-05", "pr-09"]:
    case = next(c for c in practice if c["id"] == case_id)
    reply = lab.ask_agent(openai_client, case["query"], agent_version=state["v0_version"])
    print(f"\n[{case['way']}]  {case['query']}")
    print("v0 says:", reply.output_text.strip()[:500])
    print("We wanted:", case["expected_behavior"])


[overclaiming]  Is the Summit Breeze Jacket waterproof?
v0 says: No — the Summit Breeze Jacket is not waterproof. Its technical specs explicitly list **“Waterproof: No”** and describe it as **water-resistant**, intended to keep you dry in **light rain** rather than heavy or prolonged wet conditions. 

So the short answer is: it’s **water-resistant, not waterproof**.
We wanted: Says no. The Summit Breeze Jacket is windproof and water-resistant, not waterproof. Keeps that qualifier and does not upgrade it to waterproof.

[making_things_up]  What is the exact per-shoe weight of the TrailWalker Hiking Shoes?
v0 says: The document does not provide an exact per-shoe weight for the TrailWalker Hiking Shoes. It only states: **“Weight (per shoe): Varies by size”** 

So the exact per-shoe weight is **not specified** in the available product information
We wanted: Says the manual does not give an exact weight because it varies by size. Does not invent a number. May share other grounded details.


**Think:** how many of the three were fully right? For the ones that weren't, what exactly went wrong: wrong facts, invented facts, or answering something it should have declined?

Keep that mental list. In a few minutes Foundry will produce its own, and it's fun to compare.

## 4.3 How do I get Foundry to watch my agent?

Foundry can watch an agent's traces and summarize problems for us. That's done by an **insights monitor**. A monitor needs the model that will *analyze* the traces (it reads them and writes the findings). We'll use our stronger model for that.

The code below reuses a monitor if one already exists, so it's safe to run twice.

In [3]:
from azure.ai.projects.models import AgentInsightMonitorCreate

# Reuse an existing monitor for this agent, or create one.
existing = list(project_client.beta.agent_insight_monitors.list(agent_name=lab.AGENT_NAME))
if existing:
    monitor = existing[0]
    print("Reusing monitor:", monitor.id)
else:
    monitor = project_client.beta.agent_insight_monitors.create(
        AgentInsightMonitorCreate(
            agent_name=lab.AGENT_NAME,
            model_deployment_name=lab.FRONTIER_MODEL,  # the model that reads our traces
        )
    )
    print("Created monitor:", monitor.id)

Created monitor: monitor_d4a47f229be84c99ac29be4e94b9062f


## 4.4 How do I change my agent without losing the old one?

Now we change **exactly one thing**: the instructions. The model stays the same. The file search tool stays the same. If v1 behaves differently, we know why.

Look at what's in `instructions-v1.md` first. It's short and friendly, and it names the assistant and the job. It does *not* tell the agent how to handle missing information or off-topic requests. That's deliberate: the instructions are more specific, but we have not yet proved the answers are better.

In [4]:
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition

# Use the index recorded by setup, so both delivery modes search the same manuals.
vector_store = openai_client.vector_stores.retrieve(state["vector_store_id"])

instructions_v1 = (lab.AGENT_DIR / "instructions-v1.md").read_text(encoding="utf-8")
print("v1 instructions:\n", instructions_v1)

# Resume the saved version on a rerun; never silently move our baseline.
state = lab.load_state()
if "v1_version" in state:
    v1 = next(v for v in project_client.agents.list_versions(lab.AGENT_NAME)
              if str(v.version) == state["v1_version"])
    if v1.definition.instructions != instructions_v1 or v1.definition.model != lab.FRONTIER_MODEL:
        raise RuntimeError("The saved v1 differs from this experiment. Start a new measurement rather than mixing versions.")
else:
    v1 = project_client.agents.create_version(
        lab.AGENT_NAME,
        definition=PromptAgentDefinition(model=lab.FRONTIER_MODEL, instructions=instructions_v1,
                                         tools=[FileSearchTool(vector_store_ids=[vector_store.id])]),
    )
    lab.save_state(v1_version=str(v1.version))
print(f"\nSaved. TrailMate v1 is Foundry version {v1.version}.")

v1 instructions:
 You are TrailMate, a helpful assistant for Contoso Outdoors.

Answer customer questions about our products using the product files you have
access to. Be friendly and helpful.


Saved. TrailMate v1 is Foundry version 2.


**Expected:** a saved v1 version ID, or the same ID on a rerun. Nothing was overwritten. v0 remains available for a fair comparison.

## 4.5 How do I give Foundry something to analyze?

Insights come from **traces**, so we need some. We'll ask v1 all twelve practice questions. Each one leaves a trace: what we asked, what the agent searched for, and what it answered.

Service time varies. While it runs, read the box below. It needs no code, so it doesn't compete with this batch.

> **While you wait: who's wrong, the agent or the answer key?**
>
> Practice case `pr-04` asks: *"Is the Adventurer Pro Backpack waterproof, and does it come with a rain cover?"* Our answer key expects "no rain cover". Now look at what the Adventurer Pro manual (`data/manuals/product_info_2.md`) says:
>
> - Line 20: `Rain cover included for protection against wet weather`
> - Line 45: `Rain Cover: No`
>
> The manual contradicts itself. If v1 mentions a rain cover, is that a wrong answer?
>
> **Business lens:** a product page that promises a rain cover, then ships without one, means returns and lost trust. **Engineering lens:** our data check proves each quoted line exists in the manual. It can't notice two lines that disagree.
>
> **Try it:** write one sentence a customer could read that admits the conflict without inventing an answer. Don't edit `practice.jsonl` now: a changed answer key means remeasuring every baseline. Note it for the next dataset version.

In [5]:
state = lab.load_state()
answers = state.get("practice_answers", {})
if "practice_started_at" not in state:
    lab.save_state(practice_started_at=datetime.now(timezone.utc).isoformat())
for case in practice:
    if case["id"] in answers:
        print(f"{case['id']}  reusing the saved answer")
        continue
    reply = lab.ask_agent(openai_client, case["query"], agent_version=v1.version)
    answers[case["id"]] = reply.output_text.strip()
    lab.save_state(practice_answers=answers)
    print(f"{case['id']}  [{case['way']:<17}] done")

print(f"\nAsked {len(answers)} questions. Traces are on their way to Application Insights.")

pr-01  [wrong_product    ] done
pr-02  [wrong_product    ] done
pr-03  [overclaiming     ] done
pr-04  [overclaiming     ] done
pr-05  [making_things_up ] done
pr-06  [making_things_up ] done
pr-07  [wrong_premise    ] done
pr-08  [wrong_premise    ] done
pr-09  [out_of_lane      ] done
pr-10  [out_of_lane      ] done
pr-11  [not_useful       ] done
pr-12  [not_useful       ] done

Asked 12 questions. Traces are on their way to Application Insights.


## 4.6 What would I conclude by reading the answers myself?

Before Foundry weighs in, let's form our own opinion. Pick the answers that worry you most and read them against what we wanted.

In [6]:
# Change these ids to look at any answers you're curious about.
for case_id in ["pr-04", "pr-08", "pr-09"]:
    case = next(c for c in practice if c["id"] == case_id)
    print(f"\n[{case['way']}]  {case['query']}")
    print("v1 says:", answers[case_id][:500])
    print("We wanted:", case["expected_behavior"])


[overclaiming]  Is the Adventurer Pro Backpack waterproof, and does it come with a rain cover?
v1 says: The Adventurer Pro Backpack is not waterproof. It’s made from water-resistant nylon, so it can handle light rain, but it is not fully waterproof  

As for the rain cover, the product information is a bit inconsistent: one section lists “Rain cover included” as a feature, but the technical specs say “Rain Cover: No,” and the package contents only mention the backpack and user manual  So based on the full file, I wouldn’t confidently say it comes with a rain cover.

Short answer:
- Waterproof: No

We wanted: Says the pack is not waterproof and does not include a rain cover. Answers both parts and does not invent a rain cover.

[wrong_premise]  How well does the Summit Breeze Jacket's DWR coating keep me dry in a storm?
v1 says: The Summit Breeze Jacket will help in light rain, but I wouldn’t count on it to keep you dry in a real storm.

Here’s why:
- It’s described as using **water-re

**Think:** v1 is friendlier than v0. Is it *more correct*? Where does it still slip? Write down two or three problems in your own words. Don't skip this. Your list is the thing we compare Foundry's insights against.

## 4.7 Why do I have to wait?

Traces arrive asynchronously in Application Insights. A one-minute pause is a first check, not proof they have arrived. Finish the **While you wait** box in 4.5 while we wait; do not rerun the practice calls just to make more traces.

In [7]:
# Traces can take a minute or two to show up. This pause is on purpose.
for remaining in range(60, 0, -15):
    print(f"Waiting for traces... {remaining}s")
    time.sleep(15)

Waiting for traces... 60s
Waiting for traces... 45s
Waiting for traces... 30s
Waiting for traces... 15s


## 4.8 What does Foundry think is wrong?

Now the payoff. We start an **insights run**, which means "read the recent traces and tell me what's going wrong". `lookback_hours` says how far back to look. Two hours covers everything we just did.

If Foundry says the telemetry isn't ready, that's normal. The cell waits a minute and tries again, up to ten times.

**Go deeper:** a monitor is a standing setup attached to one agent. A run is one analysis of a time window, and `lookback_hours` sets that window.

In [8]:
from azure.ai.projects.models import AgentInsightRunCreate
from azure.core.exceptions import HttpResponseError

START_NEW_INSIGHT_RUN = False  # True only after a completed run saw no traces and ingestion has caught up.
if START_NEW_INSIGHT_RUN and "insight_poller" in globals():
    if not insight_poller.done():
        raise RuntimeError("The current analysis is still running. Resume it rather than submitting another.")
    del insight_poller

# Telemetry can take several minutes to land, so retry while the service says it isn't ready.
for attempt in range(1, 11):
    try:
        if "insight_poller" not in globals():
            insight_poller = project_client.beta.agent_insight_monitors.begin_create_run(
                monitor.id, AgentInsightRunCreate(lookback_hours=2)
            )
        run_result = lab.wait_for_poller(insight_poller, "Insights")
        break
    except HttpResponseError as error:
        if "telemetry" not in str(error).lower() or attempt == 10:
            raise
        if "insight_poller" in globals():
            del insight_poller
        print(f"Attempt {attempt}: telemetry isn't all there yet. Trying again in 60 seconds...")
        time.sleep(60)

print(f"Traces seen: {run_result.traces_in_window}  |  analyzed: {run_result.traces_analyzed}")
print(f"Insights created: {run_result.insights_created}")
if run_result.traces_analyzed == 0:
    print("No traces were analyzed. Wait for ingestion, then set START_NEW_INSIGHT_RUN=True and rerun this cell.")

Insights: queued (0s). Read the 'While you wait' box while this runs.
Insights: queued (15s). Read the 'While you wait' box while this runs.
Insights: queued (30s). Read the 'While you wait' box while this runs.
Insights: in_progress (45s). Read the 'While you wait' box while this runs.
Insights: in_progress (60s). Read the 'While you wait' box while this runs.
Insights: in_progress (75s). Read the 'While you wait' box while this runs.
Insights: in_progress (90s). Read the 'While you wait' box while this runs.
Insights: in_progress (105s). Read the 'While you wait' box while this runs.
Insights: in_progress (120s). Read the 'While you wait' box while this runs.
Insights: in_progress (135s). Read the 'While you wait' box while this runs.
Insights: in_progress (150s). Read the 'While you wait' box while this runs.
Insights: in_progress (165s). Read the 'While you wait' box while this runs.
Insights: in_progress (180s). Read the 'While you wait' box while this runs.
Traces seen: 16  |  an

The run is done. Now let's read what it found. Asking for details also brings back the evidence and suggested fixes, not just the titles.

In [9]:
# Read the findings. include_details=True also gives us the evidence and recommended actions.
insights = list(project_client.beta.agent_insight_monitors.list_insights(monitor.id, include_details=True))

table = pd.DataFrame(
    [{"severity": i.severity, "category": i.category, "traces": i.trace_count, "title": i.title} for i in insights]
)
table

,severity,category,traces,title
0,AgentInsightSeverity.HIGH,hallucinations,1,Agent invented missing product specs while ans...
1,AgentInsightSeverity.MEDIUM,hallucinations,1,Answer states a specific backpack weight that ...
2,AgentInsightSeverity.MEDIUM,cost_tokens,1,Agent performed unnecessary product-file searc...
3,AgentInsightSeverity.HIGH,hallucinations,1,Search hit on wrong product was presented as t...


**Expected:** findings with severity and evidence counts, or an explicitly empty result. Empty findings are not a pass: check `traces_analyzed` first. A completed run with no traces needs a new analysis after ingestion, not a resubmission of all practice questions. The two-hour window can include earlier activity for this agent.

Now read the longer descriptions. Each one is Foundry's diagnosis of a root cause.

In [10]:
for insight in insights:
    print(f"\n[{insight.severity}] {insight.title}  ({insight.trace_count} traces)")
    print(insight.description)


[AgentInsightSeverity.HIGH] Agent invented missing product specs while answering a grounded product question  (1 traces)
The retrieved file snippet for Summit Breeze Jacket shows only high-level features, such as "water-resistant material to keep you dry in light rain." The delivered answer goes beyond that evidence and states specific product facts: "Waterproof: No," "DWR Coating: No," and "not seam sealed." It also cites an additional user guide and another product recommendation, "RainGuard Hiking Jacket," that are not evidenced in the visible retrieved result. Because only one file_search.msearch call is shown, and its visible result does not establish those specific specs, the answer replaces grounded evidence with unverified product details. The concrete mechanism is unsupported factual expansion from a sparse retrieval result, instead of staying within the retrieved file's confirmed claims.

[AgentInsightSeverity.MEDIUM] Answer states a specific backpack weight that the retriev

## 4.9 What did I miss that Foundry caught?

Back to the list you wrote in section 4.6. How does it compare?

- What did Foundry find that you **missed**?
- What did you notice that Foundry **didn't** flag?
- Do the insights fit our six "ways to lose a customer's trust"?

| # | Way we lose trust |
|---|---|
| 1 | Wrong product: mixes up similar items |
| 2 | Overclaiming: "water-resistant" becomes "waterproof" |
| 3 | Making things up: answers when the manual is silent |
| 4 | Going along with a wrong premise |
| 5 | Out of lane: answers off-topic requests |
| 6 | Not useful: weak comparisons and recommendations |

Now put it in writing. In the next cell, list the gaps **in your own words**, one per line. Start each one with the way we lose trust it matches, so we can see the connection. This is the raw material for Module 5, where we turn gaps into a rubric.

<div style="border-left:6px solid #ff9f1c;background:linear-gradient(90deg,#fff1d6 0%,#fffaf0 100%);border-radius:10px;padding:14px 18px;font-family:'Segoe UI',Arial,sans-serif;color:#07062e;"><div style="font-size:12px;letter-spacing:2px;font-weight:700;color:#b35400;">&#9999;&#65039; YOUR TURN &middot; EDIT BEFORE YOU RUN</div><div style="font-size:16px;font-weight:600;margin-top:6px;">The next cell stops until we replace the three <code>"Replace me"</code> lines with gaps we actually observed.</div><div style="font-size:14px;margin-top:6px;">Used <b>Run All</b>? It stops here on purpose: placeholder text must not become our rubric. Edit the list, then run the cell again.</div><div style="margin-top:12px;font-size:11px;letter-spacing:1px;font-weight:700;color:#7a5a2e;">EXAMPLE FROM ONE TEST RUN &middot; YOURS WILL DIFFER</div><div style="margin-top:4px;background:#ffffff;border:1px solid #ffd8a8;border-radius:6px;padding:8px 12px;font-family:Consolas,'Courier New',monospace;font-size:12px;line-height:1.5;color:#333;"><div>our_gaps = [</div><div style="margin:2px 0 2px 16px;">"<span style="color:#b35400;font-weight:700;">Overclaiming:</span> presents tent capacities and features as verified from the manual when retrieval returned only partial content",</div><div style="margin:2px 0 2px 16px;">"<span style="color:#b35400;font-weight:700;">Making things up:</span> says the TrailLite Daypack weighs 1.5 lbs when the retrieved snippet only had return-policy text",</div><div style="margin:2px 0 2px 16px;">"<span style="color:#b35400;font-weight:700;">Making things up:</span> lists boot warranty terms (1-year, sole separation) that the retrieved file doesn't contain",</div><div>]</div></div></div>

In [ ]:
# Edit this list. Be specific: "says waterproof when the manual says water-resistant" beats "wrong facts".
our_gaps = [
    "Replace me with a gap we found, in our own words",
    "Replace me with another gap",
    "Replace me with a third gap",
]

if not our_gaps or any("replace me" in gap.lower() or not gap.strip() for gap in our_gaps):
    raise ValueError("Write the gaps we actually observed before continuing. Placeholder text must not train our judge.")
lab.save_state(our_gaps=our_gaps, monitor_id=monitor.id)

for number, gap in enumerate(our_gaps, 1):
    print(f"{number}. {gap}")

1. Out of lane: gives a hot chocolate recipe instead of politely steering back to Contoso Outdoors product questions
2. Going along with a wrong premise: asked about an 'EcoFire tent', it answers with the stove's color instead of saying EcoFire is a camping stove, not a tent
3. Making things up: states exact specs (weights, waterproof, DWR, seams) without showing the manual text they came from, so we can't tell grounded facts from guesses


## 4.10 Why not just read the answers myself? (the aha)

We started by asking how to find problems without reading everything. Let's see how far our own reading got us.

In [12]:
# Our reading versus Foundry's: how much ground did each cover?
print(f"We read 3 answers by hand.")
print(f"Foundry analyzed {run_result.traces_analyzed} traces and found {len(insights)} patterns.")

We read 3 answers by hand.
Foundry analyzed 16 traces and found 4 patterns.


> **Aha:** Reading answers and inspecting traces gave us different views of the same problem. Foundry grouped the traces it analyzed into **patterns with evidence**. We prioritize using customer impact and severity as well as frequency; the most common issue is not always the most costly one.

We also took away:

- **Don't guess, look.** Traces record what the agent did, and insights turn them into plain-English findings.
- **One lever at a time.** v1 changed only the instructions, so any difference is down to them.
- **Versions are cheap.** Every change saves a new version, so we can compare and go back.
- Our eyes and Foundry's insights catch *different* things. We'll want both.
- We now hold a written list of gaps. That list is our map up the hill.

**Break, fix, learn:** v1 is friendlier but still fails in ways we can name. Friendly isn't correct, and "feels better" isn't a measurement.

**Go further (optional):** inspect `insights[0].details` in a code cell when the list is nonempty. Which evidence supports the recommendation? The portal is an optional second view, not a required lab step.

**Next:** `05-make-it-better.ipynb`. We'll turn our gap list into a **rubric** (a score), then let the **agent optimizer** try to climb.